# DFG Re=20 score-only recovery - Kaggle T4x2

This notebook consumes completed output from rodrimandub/af-pikans-dfg-re20-competitive-pilot-t4x2 through kernel_sources. It scores the ten existing final checkpoints and does not train or tune. It requires the campaign summary, source hardware metadata, all ten checkpoints and training summaries, and five saved paired pools under /kaggle/input. It can also discover and safely extract the source dfg-artifacts.zip into /kaggle/working.

Scoring reproduces the embedded campaign QoI and held-out diagnostics, adds momentum-x, momentum-y and continuity RMS inside the official near-cylinder and wake masks using saved held-out points, and reports cylinder no-slip error. It records 512-to-1024 cylinder quadrature sensitivity, per-endpoint timings and CUDA peak memory, per-seed paired summaries, source/checkpoint/pool hashes, and environment telemetry.

FEATFLOW values are QoI anchors only. No converged FEM field is present, so field_relative_l2 remains null and scientific_accepted remains false. No checkpoint is selected by score. Internet is disabled, no packages are installed, and the first code cell arms a 1,680-second hard watchdog within the 1,800-second planned Kaggle session cap.

In [ ]:
import time
SESSION_STARTED = time.perf_counter()
import os, signal, threading, json, csv, hashlib, base64, shutil, zipfile, importlib.util, sys
from pathlib import Path, PurePosixPath
from concurrent.futures import ThreadPoolExecutor, as_completed

WATCHDOG_SECONDS, SOFT_SECONDS, QUOTA_CAP_SECONDS = 1680.0, 1580.0, 1800.0
DEADLINE = SESSION_STARTED + SOFT_SECONDS
def hard_stop(): os.kill(os.getpid(), signal.SIGTERM)
watchdog = threading.Timer(WATCHDOG_SECONDS, hard_stop)
watchdog.daemon = True
watchdog.start()
RUNNER_PAYLOAD = "..."
SESSION_DIR = Path("/kaggle/working/dfg-re20-score-only") / ("run-" + str(time.time_ns()))
RESULTS_DIR = SESSION_DIR / "scores"
SESSION_DIR.mkdir(parents=True, exist_ok=False)
RESULTS_DIR.mkdir(parents=True, exist_ok=False)

def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1048576), b""): h.update(block)
    return h.hexdigest()
def check_time(label):
    if time.perf_counter() >= DEADLINE: raise TimeoutError("score-only soft deadline reached: " + label)

SEEDS=(501,502,503,504,505)
ARMS=("DFG-RBF-PIKAN","DFG-Tanh-MLP")
EXPECTED={(a,s) for s in SEEDS for a in ARMS}
POOL_KEYS=("interior","inlet","lower_wall","upper_wall","cylinder","outlet","heldout",
           "validation_inlet","validation_lower_wall","validation_upper_wall","validation_cylinder")

def campaign_roots(bases):
    found=set()
    for base in bases:
        if base.exists():
            found.update(p.parent.resolve() for p in base.rglob("campaign_results.json"))
    return sorted(found)
def missing(root):
    out=[n for n in ("campaign_results.json","hardware.json") if not (root/n).is_file()]
    for seed in SEEDS:
        if not (root/"pools"/f"seed-{seed}.npz").is_file(): out.append(f"pools/seed-{seed}.npz")
        for arm in ARMS:
            d=root/arm/f"seed-{seed}"
            for n in ("checkpoint-final.pt","training_summary.json"):
                if not (d/n).is_file(): out.append(f"{arm}/seed-{seed}/{n}")
    return out
def fail(msg, details=None):
    write_json(SESSION_DIR/"artifact_discovery_failure.json",{
        "status":"blocked_missing_source_artifact","source_kernel":SOURCE_KERNEL,
        "message":msg,"details":details,"training_repeated":False,
        "field_relative_l2":None,"field_reference_status":"missing_converged_FEM_field",
        "scientific_accepted":False})
    raise FileNotFoundError(msg)
def extract_relevant(zip_path, target):
    kept=[]
    with zipfile.ZipFile(zip_path) as z:
        for item in z.infolist():
            member=PurePosixPath(item.filename); parts=member.parts
            if not parts or item.is_dir(): continue
            name=parts[-1]
            wanted=(name in ("campaign_results.json","hardware.json","session_summary.json",
                             "checkpoint-final.pt","training_summary.json","training_history.jsonl")
                    or (name.startswith("seed-") and name.endswith(".npz") and "pools" in parts))
            if not wanted: continue
            if member.is_absolute() or ".." in parts: raise RuntimeError("unsafe archive path: "+item.filename)
            dest=(target.joinpath(*parts)).resolve()
            if os.path.commonpath((str(target.resolve()),str(dest))) != str(target.resolve()):
                raise RuntimeError("unsafe archive path: "+item.filename)
            dest.parent.mkdir(parents=True,exist_ok=True)
            with z.open(item) as src, dest.open("wb") as dst: shutil.copyfileobj(src,dst)
            kept.append(str(dest))
    return kept

def region_residuals(model, points, device):
    x,y=points[:,0],points[:,1]
    r=np.sqrt((x-.2)**2+(y-.2)**2)
    fluid=(r>.05)&(x>=0)&(x<=2.2)&(y>=0)&(y<=.41)
    masks={"near_cylinder":fluid&(r<.10),
           "wake":fluid&(x>=.25)&(x<=1.0)&(np.abs(y-.2)<=.1)}
    result={}
    for region,mask in masks.items():
        subset=points[mask]
        if not len(subset):
            result[region]={"status":"empty_region","point_count":0}; continue
        sums={"momentum_x":0.0,"momentum_y":0.0,"continuity":0.0}
        for start in range(0,len(subset),256):
            check_time("critical-region residuals")
            xy=torch.as_tensor(subset[start:start+256],dtype=torch.float32,device=device).requires_grad_(True)
            residuals=campaign.residual_components(model(xy),xy)
            for name,value in residuals.items(): sums[name]+=float(value.detach().square().sum().cpu())
        n=len(subset)
        result[region]={"status":"scored","point_count":n,
            "mask":"fluid 0.05<r_center<0.10" if region=="near_cylinder"
                   else "fluid 0.25<=x<=1.0 and abs(y-0.2)<=0.1",
            "momentum_x_rms_normalized":(sums["momentum_x"]/n)**.5,
            "momentum_y_rms_normalized":(sums["momentum_y"]/n)**.5,
            "continuity_rms_normalized":(sums["continuity"]/n)**.5}
    return result

def score_model(item, model, device):
    arm,seed=item["arm"],item["seed"]; phases={}
    def phase(key,fn):
        torch.cuda.synchronize(device); t=time.perf_counter(); value=fn(); torch.cuda.synchronize(device)
        phases[key]=time.perf_counter()-t
        return value
    started=time.perf_counter()
    q512=phase("qoi512_seconds",lambda:campaign.qoi_for_model(model,device,512,DEADLINE))
    q1024=phase("qoi1024_seconds",lambda:campaign.qoi_for_model(model,device,1024,DEADLINE))
    diag=phase("heldout_and_boundary_seconds",lambda:campaign.heldout_diagnostics(
        model,item["pools"]["heldout"],item["pools"],device,DEADLINE))
    regions=phase("critical_region_seconds",lambda:region_residuals(model,item["pools"]["heldout"],device))
    ref=campaign.FEATFLOW
    errors={"drag_relative_error_vs_featflow":abs(q512["drag_coefficient"]-ref["drag_coefficient"])/ref["drag_coefficient"],
            "lift_absolute_error_vs_featflow":abs(q512["lift_coefficient"]-ref["lift_coefficient"]),
            "pressure_difference_relative_error_vs_featflow":abs(q512["pressure_difference"]-ref["pressure_difference"])/ref["pressure_difference"]}
    changes={k:abs(q1024[k]-q512[k]) for k in ("drag_coefficient","lift_coefficient","pressure_difference")}
    return {"status":"score_only_complete","arm":arm,"seed":seed,"model_id":model.model_id,
        "qois":q512,"featflow_reference":ref,"featflow_source":campaign.FEATFLOW_URL,
        "qoi_errors":errors,"quadrature_512_to_1024_absolute_change":changes,
        "diagnostics":diag,"critical_region_residuals":regions,
        "cylinder_velocity_rmse_over_umean":diag.get("cylinder_velocity_rmse_over_umean"),
        "field_relative_l2":None,"field_reference_status":"missing_converged_FEM_field",
        "scientific_accepted":False,"training_repeated":False,"confirmatory_scoring_only":True,
        "heldout_test_scored":True,"checkpoint_selection":"final checkpoint only; no score-based selection",
        "source_checkpoint_sha256":item["checkpoint_hash"],"saved_pool_sha256":item["pool_hash"],
        "source_training_status":item["source_status"],"source_training_summary":item["training_summary"],
        "device":torch.cuda.get_device_name(device),"cuda_device_index":int(device.index),"dtype":"float32",
        "phase_wall_seconds":phases,"total_scoring_wall_seconds":time.perf_counter()-started,
        "cuda_peak_memory_allocated_bytes":int(torch.cuda.max_memory_allocated(device)),
        "cuda_peak_memory_reserved_bytes":int(torch.cuda.max_memory_reserved(device))}

SOURCE_KERNEL="rodrimandub/af-pikans-dfg-re20-competitive-pilot-t4x2"
try:
    import numpy as np
    import scipy
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count()!=2:
        raise RuntimeError("Expected exactly two visible CUDA devices for T4x2; CPU fallback is disabled.")
    gpu_names=[torch.cuda.get_device_name(i) for i in range(2)]
    if any("T4" not in name.upper() for name in gpu_names): raise RuntimeError("Expected two Tesla T4 devices.")
    torch.set_num_threads(2); torch.set_default_dtype(torch.float32); torch.backends.cuda.matmul.allow_tf32=False

    runner_path=SESSION_DIR/"run_campaign_source.py"
    runner_path.write_bytes(base64.b64decode(RUNNER_PAYLOAD)); runner_hash=sha256(runner_path)
    spec=importlib.util.spec_from_file_location("dfg_scoring_source",runner_path)
    campaign=importlib.util.module_from_spec(spec); spec.loader.exec_module(campaign)

    input_root=Path("/kaggle/input")
    if not input_root.is_dir(): fail("No /kaggle/input; attach the completed source kernel outputs.")
    roots=campaign_roots([input_root]); complete=[r for r in roots if not missing(r)]; extracted=[]
    if not complete:
        arcdir=SESSION_DIR/"source-archive-extracts"
        for archive_path in sorted(input_root.rglob("*.zip")):
            if time.perf_counter()>=DEADLINE: break
            try:
                with zipfile.ZipFile(archive_path) as z:
                    has_summary=any(PurePosixPath(n).name=="campaign_results.json" for n in z.namelist())
                if has_summary:
                    dest=arcdir/(archive_path.stem+"-"+sha256(archive_path)[:12])
                    extracted.extend(extract_relevant(archive_path,dest))
            except zipfile.BadZipFile: continue
        roots=campaign_roots([input_root,arcdir]); complete=[r for r in roots if not missing(r)]
    if len(complete)!=1:
        fail("Expected one complete DFG campaign with all ten checkpoints, five pools, summaries and hardware.",
             {"input_entries":[str(p) for p in list(input_root.iterdir())[:40]],
              "candidate_roots":[{"root":str(r),"missing":missing(r)} for r in roots],
              "extracted_source_files":extracted[:100]})
    campaign_root=complete[0]
    source=json.loads((campaign_root/"campaign_results.json").read_text(encoding="utf-8"))
    hardware=json.loads((campaign_root/"hardware.json").read_text(encoding="utf-8"))
    if source.get("source_sha256")!=runner_hash:
        fail("Embedded runner SHA does not match the campaign source SHA.",
             {"embedded":runner_hash,"campaign":source.get("source_sha256")})
    if source.get("scientific_accepted") is not False: raise RuntimeError("Source acceptance must remain false.")
    if "T4" not in str(hardware.get("device","")).upper(): raise RuntimeError("Source hardware is not Tesla T4.")
    status={}
    for row in source.get("results",[]):
        if isinstance(row,dict) and "arm" in row and "seed" in row:
            status[(row["arm"],int(row["seed"]))]=row.get("status")
    if set(status)!=EXPECTED or any(status[k]!="score_budget_exhausted" for k in EXPECTED):
        fail("Source output must have exactly ten score_budget_exhausted endpoints.",
             {f"{a}|{s}":v for (a,s),v in status.items()})
    for key,value in campaign.FEATFLOW.items():
        if key not in source.get("featflow_qois",{}) or abs(float(source["featflow_qois"][key])-float(value))>1e-12:
            raise RuntimeError("FEATFLOW values do not match the embedded source.")
    if source.get("field_reference_status")!="missing_converged_FEM_field":
        raise RuntimeError("Unexpected field-reference status; this notebook does not supply an FEM field.")

    endpoints=[]; pool_hashes={}; train_summaries={}; histories={}
    for seed in SEEDS:
        pool_path=campaign_root/"pools"/f"seed-{seed}.npz"
        with np.load(pool_path,allow_pickle=False) as z:
            absent=sorted(set(POOL_KEYS)-set(z.files))
            if absent: raise RuntimeError(f"Saved pool seed {seed} lacks keys: {absent}")
            pools={k:z[k].copy() for k in POOL_KEYS}
        if pools["interior"].shape!=(16384,2) or pools["heldout"].shape!=(8192,2):
            raise RuntimeError(f"Unexpected interior/heldout shape for saved pool seed {seed}.")
        for k in ("validation_inlet","validation_lower_wall","validation_upper_wall","validation_cylinder"):
            if pools[k].shape!=(512,2): raise RuntimeError(f"Unexpected {k} shape for seed {seed}.")
        ph=sha256(pool_path); pool_hashes[str(seed)]=ph
        for arm in ARMS:
            d=campaign_root/arm/f"seed-{seed}"; key=f"{arm}|{seed}"
            ts=json.loads((d/"training_summary.json").read_text(encoding="utf-8"))
            train_summaries[key]=ts
            history=d/"training_history.jsonl"
            if history.is_file():
                lines=[s for s in history.read_text(encoding="utf-8").splitlines() if s.strip()]
                histories[key]=json.loads(lines[-1]) if lines else None
            cp=d/"checkpoint-final.pt"
            endpoints.append({"arm":arm,"seed":seed,"path":cp,"checkpoint_hash":sha256(cp),
                              "pool_path":pool_path,"pool_hash":ph,"pools":pools,
                              "source_status":status[(arm,seed)],"training_summary":ts})
    session_info=[]
    for candidate in input_root.rglob("session_summary.json"):
        try:
            entry=json.loads(candidate.read_text(encoding="utf-8"))
            if entry.get("runner_sha256")==runner_hash: session_info.append({"path":str(candidate),"summary":entry})
        except Exception: pass

    manifest={"status":"score_only_running","source_kernel":SOURCE_KERNEL,"source_campaign_root":str(campaign_root),
        "source_campaign_wall_seconds":source.get("campaign_wall_seconds"),"source_runner_sha256":runner_hash,
        "source_hardware":hardware,"source_session_summary":session_info,"source_status_by_endpoint":{f"{a}|{s}":status[(a,s)] for a,s in sorted(EXPECTED)},
        "training_summaries":train_summaries,"training_history_last_record":histories,
        "pool_sha256_by_seed":pool_hashes,"featflow_reference":campaign.FEATFLOW,"featflow_source":campaign.FEATFLOW_URL,
        "field_relative_l2":None,"field_reference_status":"missing_converged_FEM_field","scientific_accepted":False,
        "training_repeated":False,"confirmatory_scoring_only":True,"paired_scoring":"two endpoints concurrently, one per visible T4",
        "gpu_names":gpu_names,"dtype":"float32","watchdog_seconds_from_first_code_cell":WATCHDOG_SECONDS,
        "score_soft_deadline_seconds_from_first_code_cell":SOFT_SECONDS,"session_quota_cap_seconds_including_setup":QUOTA_CAP_SECONDS,
        "internet_enabled":False}
    write_json(SESSION_DIR/"score_only_manifest.json",manifest)
    results=[]; seed_summaries=[]
    for seed in SEEDS:
        pair=[e for e in endpoints if e["seed"]==seed]
        if time.perf_counter()>=DEADLINE:
            for e in pair: results.append({"arm":e["arm"],"seed":seed,"status":"score_only_budget_exhausted_before_endpoint","field_relative_l2":None,"scientific_accepted":False})
            continue
        models=[]
        for index,e in enumerate(pair):
            device=torch.device(f"cuda:{index}"); torch.cuda.reset_peak_memory_stats(device)
            cp=torch.load(e["path"],map_location=device,weights_only=False)
            model=campaign.build_model(e["arm"],seed).to(device=device,dtype=torch.float32)
            model.load_state_dict(cp["state_dict"],strict=True); model.eval()
            models.append((e,model,device))
        pair_started=time.perf_counter(); pair_rows=[]
        with ThreadPoolExecutor(max_workers=2) as pool:
            future_map={pool.submit(score_model,e,model,device):(e,device) for e,model,device in models}
            for future in as_completed(future_map):
                e,device=future_map[future]; out=RESULTS_DIR/e["arm"]/f"seed-{seed}"; out.mkdir(parents=True,exist_ok=False)
                try:
                    row=future.result()
                    row.update({"source_checkpoint_path":str(e["path"]),"source_pool_path":str(e["pool_path"]),
                        "reference_status":"FEATFLOW QoI anchors only; no converged FEM field","reference_accepted":False,
                        "source_scoring_runner_sha256":runner_hash})
                    write_json(out/"score.json",row)
                except Exception as exc:
                    row={"arm":e["arm"],"seed":seed,"status":"score_failed_preserved",
                        "exception":f"{type(exc).__name__}: {exc}","source_checkpoint_sha256":e["checkpoint_hash"],
                        "saved_pool_sha256":e["pool_hash"],"source_training_status":e["source_status"],
                        "field_relative_l2":None,"scientific_accepted":False,"reference_accepted":False}
                    write_json(out/"score_failure.json",row)
                pair_rows.append(row); results.append(row)
                write_json(SESSION_DIR/"score_only_progress.json",{**manifest,"results_completed":len(results),"results":results})
        del models
        by_arm={r.get("arm"):r for r in pair_rows}
        pair_summary={"seed":seed,"status":"paired_scored" if all(by_arm.get(a,{}).get("status")=="score_only_complete" for a in ARMS) else "paired_partial_or_failed",
                      "pool_sha256":pool_hashes[str(seed)],"descriptive_only":True,"tanh_minus_rbf":{},
                      "pair_scoring_wall_seconds":time.perf_counter()-pair_started}
        if all(by_arm.get(a,{}).get("status")=="score_only_complete" for a in ARMS):
            for metric in ("drag_coefficient","lift_coefficient","pressure_difference"):
                pair_summary["tanh_minus_rbf"][metric]=by_arm["DFG-Tanh-MLP"]["qois"][metric]-by_arm["DFG-RBF-PIKAN"]["qois"][metric]
        seed_summaries.append(pair_summary); write_json(SESSION_DIR/f"seed-{seed}-summary.json",pair_summary)

    success=[r for r in results if r.get("status")=="score_only_complete"]
    final_status="score_only_complete" if len(success)==10 else "score_only_partial_failure"
    report={**manifest,"status":final_status,"campaign_wall_seconds_from_first_code_cell":time.perf_counter()-SESSION_STARTED,
        "successful_endpoints":len(success),"expected_endpoints":10,"results":results,"per_seed_summaries":seed_summaries,
        "field_relative_l2":None,"field_reference_status":"missing_converged_FEM_field","scientific_accepted":False}
    write_json(SESSION_DIR/"score_only_results.json",report)
    write_json(SESSION_DIR/"summary_by_seed.json",{"status":final_status,"seeds":seed_summaries})
    columns=("arm","seed","status","drag_coefficient","lift_coefficient","pressure_difference",
        "drag_relative_error_vs_featflow","lift_absolute_error_vs_featflow","pressure_difference_relative_error_vs_featflow",
        "heldout_momentum_x_rms","heldout_momentum_y_rms","heldout_continuity_rms",
        "near_cylinder_momentum_x_rms_normalized","near_cylinder_momentum_y_rms_normalized","near_cylinder_continuity_rms_normalized",
        "wake_momentum_x_rms_normalized","wake_momentum_y_rms_normalized","wake_continuity_rms_normalized",
        "cylinder_velocity_rmse_over_umean","total_scoring_wall_seconds","device","cuda_device_index","scientific_accepted")
    with (SESSION_DIR/"endpoint_scores.csv").open("w",newline="",encoding="utf-8") as f:
        w=csv.DictWriter(f,fieldnames=columns,extrasaction="ignore"); w.writeheader()
        for r in results:
            row={k:r.get(k) for k in ("arm","seed","status","total_scoring_wall_seconds","device","cuda_device_index","scientific_accepted")}
            row.update(r.get("qois",{})); row.update(r.get("qoi_errors",{}))
            d=r.get("diagnostics",{}); row.update({k:d.get(k) for k in ("heldout_momentum_x_rms","heldout_momentum_y_rms","heldout_continuity_rms","cylinder_velocity_rmse_over_umean")})
            for region in ("near_cylinder","wake"):
                for component in ("momentum_x","momentum_y","continuity"):
                    row[f"{region}_{component}_rms_normalized"]=r.get("critical_region_residuals",{}).get(region,{}).get(f"{component}_rms_normalized")
            w.writerow(row)
    telemetry={"status":final_status,"source_runner_sha256":runner_hash,"campaign_wall_seconds":time.perf_counter()-SESSION_STARTED,
        "quota_cap_seconds_including_setup":QUOTA_CAP_SECONDS,"watchdog_seconds_from_first_code_cell":WATCHDOG_SECONDS,
        "soft_deadline_seconds_from_first_code_cell":SOFT_SECONDS,"python":sys.version,"numpy":np.__version__,
        "scipy":scipy.__version__,"torch":torch.__version__,"cuda":torch.version.cuda,"visible_gpus":gpu_names,
        "source_hardware":hardware,"endpoint_telemetry":[{k:r.get(k) for k in
        ("arm","seed","source_training_summary","source_checkpoint_sha256","saved_pool_sha256","phase_wall_seconds",
         "total_scoring_wall_seconds","device","cuda_device_index","cuda_peak_memory_allocated_bytes","cuda_peak_memory_reserved_bytes")}
        for r in results]}
    write_json(SESSION_DIR/"telemetry.json",telemetry)
    for i in range(2): torch.cuda.synchronize(torch.device(f"cuda:{i}"))
    print(json.dumps({"status":final_status,"successful_endpoints":len(success),"expected_endpoints":10,
        "session_output":str(SESSION_DIR),"wall_seconds":time.perf_counter()-SESSION_STARTED,
        "field_relative_l2":None,"field_reference_status":"missing_converged_FEM_field",
        "scientific_accepted":False,"gpu_names":gpu_names},indent=2))
    if len(success)!=10: raise RuntimeError("Score-only recovery ended partially; completed scores were preserved.")
finally:
    watchdog.cancel()